# Saving the Infidels

In this notebook we want so solve a famous search problem, which is usually known as the
[missionaries and cannibals problem](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem):
Three missionaries and three infidels have to cross a river in order to get to a church where the infidels can be baptized.  In order to cross the river, they have to take a small boat that can take at most two passengers.  If at any moments at any shore there are more infidels than missionaries, then the missionaries have a problem, since the infidels have a diet that includes human flesh.

We will encode this problem as a *constraint satisfaction problem* that is solved with `Z3`.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossings of the river.  We use the following variables:
* $\texttt{M}i$ for $i\in\{0,\cdots,n\}$ is the number of missionaries on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is the number of infidels on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{B}i$ for $i\in\{0,\cdots,n\}$ is the number of boats on the western shore after the 
  $i^{\textrm{th}}$ crossing.

In [1]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

## Auxiliary Functions

The symbolic transition system uses 3 variables.
* `M` is the number of missionaries on the left side of the river,
* `C` is the number of infidels on the left side of the river,
* `B` is the number of boats on the left side of the river.

In [3]:
let M = Int "M"
let C = Int "C"
let B = Int "B"

The function `start` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns a formula that specifies that everybody is on the western shore.

In [4]:
let start (M: IntExpr) (C: IntExpr) (B: IntExpr) : BoolExpr =
    And [ M .= 3; C .= 3; B .= 1 ]

In [5]:
start M C B

M = 3 ∧ C = 3 ∧ B = 1

The function `goal` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns a formula that specifies that everybody is on the eastern shore.

In [6]:
let goal (M: IntExpr) (C: IntExpr) (B: IntExpr) : BoolExpr =
    And [ M .= 0; C .= 0; B .= 0 ]

The function `invariant` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns a list of formulas that are `true` if there is no problem on either shore of the river.  There is no problem if any of the following conditions is true:
* There are no missionaries on the western side of the shore, i.e. 
  $\texttt{M} = 0$.  
  Then all missionaries are on the eastern side of the shore.
* All missionaries are on the western side of the shore, i.e. $\texttt{M} = 3$.
  Then there are no missionaries on the eastern side of the shore.
* The number of missionaries on the western side is the same as the number of 
  infidels on that side, i.e. $\texttt{M} = \texttt{C}$.  Then the numbers of 
  missionaries and infidels have to match on the eastern shore as well.

Furthermore, the formulas specify the ranges of the variables.

In [7]:
let invariant (M: IntExpr) (C: IntExpr) (B: IntExpr) : BoolExpr list =
    [ Or [ M .= 0; M .= 3; M .= C ]
      0 .<= M; M .<= 3
      0 .<= C; C .<= 3
      0 .<= B; B .<= 1 ]

In [8]:
for constraint in invariant M C B do
    print constraint

M = 0 ∨ M = 3 ∨ M = C

0 ≤ M

M ≤ 3

0 ≤ C

C ≤ 3

0 ≤ B

B ≤ 1


input.fsx (1,5)-(1,15) parse warning The identifier 'constraint' is reserved for future use by F#

input.fsx (2,11)-(2,21) parse warning The identifier 'constraint' is reserved for future use by F#



The function `boatOK` takes 6 arguments:
* `Mα` is the number of missionaries on the western shore before the crossing.
* `Cα` is the number of infidels on the western shore before the crossing.
* `Bα` is the number of boats on the western shore before the crossing. 
* `Mβ` is the number of missionaries on the western shore after the crossing.
* `Cβ` is the number of infidels on the western shore after the crossing.
* `Bβ` is the number of boats on the western shore after the crossing.

It returns a formula that is `true` if the following holds: if the boat starts on the western shore, then it carries one or two persons to the eastern shore.

In [9]:
let boatOK (Mα: IntExpr) (Cα: IntExpr) (Bα: IntExpr) (Mβ: IntExpr) (Cβ: IntExpr) (Bβ: IntExpr) : BoolExpr =
    Implies(Bα .= 1, And [ 1 .<= Mα - Mβ + Cα - Cβ
                           2 .>= Mα - Mβ + Cα - Cβ
                           Mβ .<= Mα
                           Cβ .<= Cα ])

In [10]:
let MX = Int "MX"
let CX = Int "CX"
let BX = Int "BX"
boatOK M C B MX CX BX

B = 1 → 1 ≤ M - MX + C - CX ∧ 2 ≥ M - MX + C - CX ∧ MX ≤ M ∧ CX ≤ C

The function `transition` takes the same 6 arguments as `boatOK`.  It returns a list of formulas that are `true` if the persons starting on one shore arrive at the opposite shore.  The boat changes sides during every crossing.  If the boat starts on the western shore, `boatOK Mα Cα Bα Mβ Cβ Bβ` describes the crossing.  If it starts on the eastern shore, then the boat is on the western shore after the crossing and the crossing is described by `boatOK Mβ Cβ Bβ Mα Cα Bα`.

In [11]:
let transition (Mα: IntExpr) (Cα: IntExpr) (Bα: IntExpr) (Mβ: IntExpr) (Cβ: IntExpr) (Bβ: IntExpr) : BoolExpr list =
    [ Bβ .= 1 - Bα
      boatOK Mα Cα Bα Mβ Cβ Bβ
      boatOK Mβ Cβ Bβ Mα Cα Bα ]

In [12]:
transition M C B MX CX BX

[BX = 1 - B; B = 1 → 1 ≤ M - MX + C - CX ∧ 2 ≥ M - MX + C - CX ∧ MX ≤ M ∧ CX ≤ C; BX = 1 → 1 ≤ MX - M + CX - C ∧ 2 ≥ MX - M + CX - C ∧ M ≤ MX ∧ C ≤ CX]

The function `missionariesCSP` takes one argument:
- `n` is the number of crossings.

It tries to solve the problem with `n` crossings.  If this is possible, `Some Solution` is returned, where `Solution` is a map assigning values to the variables.  Otherwise, `None` is returned.

In [13]:
let missionariesCSP (n: int) : Map<string, int> option =
    let S  = newSolver ()
    let Ms = [| for i in 0 .. n -> Int (sprintf "M%d" i) |]
    let Cs = [| for i in 0 .. n -> Int (sprintf "C%d" i) |]
    let Bs = [| for i in 0 .. n -> Int (sprintf "B%d" i) |]
    let Constraints = [ start Ms[0] Cs[0] Bs[0]    // start state
                        goal  Ms[n] Cs[n] Bs[n]    // goal state
                        for i in 0 .. n - 1 do
                            yield! invariant Ms[i] Cs[i] Bs[i]
                            yield! transition Ms[i] Cs[i] Bs[i] Ms[i+1] Cs[i+1] Bs[i+1] ]
    S.Add(Array.ofList Constraints)
    if S.Check() = Status.SATISFIABLE then
        let m = S.Model
        Some (Map [ for (name, X) in [ ("M", Ms); ("C", Cs); ("B", Bs) ] do
                      for i in 0 .. n -> (sprintf "%s%d" name i, intValue m X[i]) ])
    else
        None

The function `findSolution` takes no arguments.  It computes a solution to the problem of saving the infidels by trying $n = 1, 3, 5, \cdots$ crossings until a solution is found.  It returns the number of crossings `n` together with the solution.

In [14]:
let findSolution () : int * Map<string, int> =
    let rec search (n: int) =
        printfn "%d" n
        match missionariesCSP n with
        | Some solution -> (n, solution)
        | None          -> search (n + 2)
    search 1

In [15]:
#!time
let n, Solution = findSolution ()

1

3

5

7

9

11

Wall time: 37.7596ms

In [16]:
n, Solution

(11, map [("B0", 1); ("B1", 0); ("B10", 1); ("B11", 0); ("B2", 1); ("B3", 0); ("B4", 1); ("B5", 0); ("B6", 1); ("B7", 0); ("B8", 1); ("B9", 0); ("C0", 3); ("C1", 2); ("C10", 2); ("C11", 0); ("C2", 2); ("C3", 0); ("C4", 1); ("C5", 1); ("C6", 2); ("C7", 2); ("C8", 3); ("C9", 1); ("M0", 3); ("M1", 2); ("M10", 0); ("M11", 0); ("M2", 3); ("M3", 3); ("M4", 3); ("M5", 1); ("M6", 2); ("M7", 0); ("M8", 0); ("M9", 0)])

The function `showSolution` takes two arguments:
- `Solution` is a map assigning values to the variables,
- `n` is the number of crossings.

It prints the solution.

In [17]:
let showSolution (Solution: Map<string, int>) (n: int) : unit =
    let rep (k: int) (s: string) = String.replicate k s
    for i in 0 .. n do
        let M = Solution[sprintf "M%d" i]
        let C = Solution[sprintf "C%d" i]
        let B = Solution[sprintf "B%d" i]
        printfn "%s%s%s%s%s" (rep M "😇") (rep C "🥷") (String.replicate 28 " ") (rep (3 - M) "😇") (rep (3 - C) "🥷")
        if i < n then
            if B = 1 then
                let MB = M - Solution[sprintf "M%d" (i+1)]
                let CB = C - Solution[sprintf "C%d" (i+1)]
                printfn "%s>>> %s%s >>>" (String.replicate 12 " ") (rep MB "😇") (rep CB "🥷")
            else
                let MB = Solution[sprintf "M%d" (i+1)] - M
                let CB = Solution[sprintf "C%d" (i+1)] - C
                printfn "%s<<< %s%s <<<" (String.replicate 12 " ") (rep MB "😇") (rep CB "🥷")

In [18]:
showSolution Solution n

😇😇😇

🥷🥷🥷

>>> 

😇

🥷

...

>>> 

🥷🥷

 >>>

😇😇😇

🥷🥷🥷